In [2]:
!pip install torchreid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=9ea57a83773691d26d4494fa3cf0f80cc29fa192e6d9090d41c8b2deb7e301d3
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid


In [4]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from sklearn.metrics import average_precision_score
import torchreid
from torchreid import utils

print("Setup selesai!")

Setup selesai!


In [3]:
# ============================================================
# Definisi 3 konfigurasi model
# ============================================================
MODEL_CONFIGS = [
    {
        "name"       : "Pretrained ImageNet",
        "num_classes": 1000,
        "weight_url" : None,
        "weight_file": None,
    },
    {
        "name"       : "Pretrained Market-1501",
        "num_classes": 751,
        "weight_url" : "https://drive.google.com/uc?id=1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA",
        "weight_file": "osnet_x1_0_market1501.pth",
    },
    {
        "name"       : "Pretrained MSMT17",
        "num_classes": 1041,
        "weight_url" : "https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M",
        "weight_file": "osnet_x1_0_msmt17.pth",
    },
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def build_model_from_config(cfg):
    model = torchreid.models.build_model(
        name='osnet_x1_0',
        num_classes=cfg["num_classes"],
        pretrained=cfg["weight_url"] is None
    )
    if cfg["weight_url"] is not None:
        utils.download_url(cfg["weight_url"], cfg["weight_file"])
        utils.load_pretrained_weights(model, cfg["weight_file"])
        print(f"Weight loaded: {cfg['weight_file']}")
    else:
        print("Pretrained ImageNet — weight ImageNet digunakan")
    model = model.to(device)
    model.eval()
    return model

print("Konfigurasi model siap!")

Device: cuda
Konfigurasi model siap!


In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
def load_images_from_folder(folder_path):
    """
    Load gambar dari folder dengan format nama: XXXX_cY_fZZZZ.jpg
    Return: list of (image_path, person_id, camera_id)
    """
    data = []
    for fname in sorted(os.listdir(folder_path)):
        if not fname.endswith('.jpg'):
            continue
        
        # Parse ID dari nama file
        # Format: XXXX_cY_fZZZZ.jpg
        parts = fname.split('_')
        try:
            person_id = int(parts[0])
            camera_id = int(parts[1][1:])  # hilangkan 'c'
        except:
            continue
        
        # Skip ID -1 (junk/distractor)
        if person_id == -1:
            continue
            
        img_path = os.path.join(folder_path, fname)
        data.append((img_path, person_id, camera_id))
    
    return data

print("Fungsi siap!")

Fungsi siap!


In [7]:
def extract_features_from_paths(data_list, model, transform, device, batch_size=32):
    """
    Ekstrak fitur dari list of (img_path, person_id, camera_id)
    Return: features, labels, camera_ids
    """
    features_list = []
    labels_list   = []
    cameras_list  = []
    
    model.eval()
    with torch.no_grad():
        for i in range(0, len(data_list), batch_size):
            batch = data_list[i:i+batch_size]
            imgs  = torch.stack([
                transform(Image.open(p).convert('RGB')) for p, _, _ in batch
            ]).to(device)
            features_list.append(model(imgs).cpu().numpy())
            labels_list.extend([x[1] for x in batch])
            cameras_list.extend([x[2] for x in batch])
            
            if i % 320 == 0:
                print(f"  Processed {i}/{len(data_list)}")
    
    return np.vstack(features_list), labels_list, cameras_list
 
print("Fungsi siap!")

Fungsi siap!


In [9]:
def evaluate_reid_reranking(query_feats, query_labels, gallery_feats, gallery_labels, k1=20, k2=6, lambda_value=0.3):
    # Hitung 3 distance matrix
    q_q_dist = compute_cosine_distance(query_feats, query_feats)
    q_g_dist = compute_cosine_distance(query_feats, gallery_feats)
    g_g_dist = compute_cosine_distance(gallery_feats, gallery_feats)
    
    # Re-ranking
    dist_matrix = utils.re_ranking(
        q_g_dist, q_q_dist, g_g_dist,
        k1=k1, k2=k2, lambda_value=lambda_value
    )
    
    num_query  = len(query_labels)
    cmc_scores = np.zeros(10)
    ap_list    = []

    for i in range(num_query):
        q_label    = query_labels[i]
        sorted_idx = np.argsort(dist_matrix[i])
        sorted_labels = [gallery_labels[j] for j in sorted_idx]
        matches    = [1 if lbl == q_label else 0 for lbl in sorted_labels]
        
        if sum(matches) == 0:
            continue
        
        for rank in range(10):
            if sum(matches[:rank+1]) > 0:
                cmc_scores[rank] += 1
        
        ap_list.append(average_precision_score(
            matches, [-d for d in dist_matrix[i][sorted_idx]]
        ))

    cmc_scores /= num_query
    mAP = np.mean(ap_list) if ap_list else 0
    return cmc_scores, mAP

print("Fungsi re-ranking siap!")

Fungsi re-ranking siap!


In [8]:
def compute_cosine_distance(qf, gf):
    q = qf / np.linalg.norm(qf, axis=1, keepdims=True)
    g = gf / np.linalg.norm(gf, axis=1, keepdims=True)
    return 1 - np.dot(q, g.T)

def evaluate_reid(query_feats, query_labels, gallery_feats, gallery_labels):
    dist_matrix = compute_cosine_distance(query_feats, gallery_feats)
    
    num_query  = len(query_labels)
    cmc_scores = np.zeros(10)
    ap_list    = []

    for i in range(num_query):
        q_label    = query_labels[i]
        sorted_idx = np.argsort(dist_matrix[i])
        sorted_labels = [gallery_labels[j] for j in sorted_idx]
        matches    = [1 if lbl == q_label else 0 for lbl in sorted_labels]
        
        if sum(matches) == 0:
            continue
        
        for rank in range(10):
            if sum(matches[:rank+1]) > 0:
                cmc_scores[rank] += 1
        
        ap_list.append(average_precision_score(
            matches, [-d for d in dist_matrix[i][sorted_idx]]
        ))

    cmc_scores /= num_query
    mAP = np.mean(ap_list) if ap_list else 0
    return cmc_scores, mAP

print("Fungsi baseline siap!")

Fungsi baseline siap!


In [10]:
dataset_root = "/kaggle/input/datasets/singh96divya/wb-wob-reid-dataset/WB_WoB-ReID"
subsets      = ["with_bag", "without_bag", "both_small", "both_large"]

# Simpan hasil semua model
all_model_results = {}
models_dict = {}
gallery_cache = {}  # ← tambah ini sebelum loop

for cfg in MODEL_CONFIGS:
    model_name = cfg["name"]
    print(f"\n{'#'*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'#'*70}")

    # Build model sesuai config
    model = build_model_from_config(cfg)

    all_model_results[model_name] = {}
    gallery_cache[model_name] = {}  # ← tambah ini

    for subset in subsets:
        print(f"\n{'='*50}")
        print(f"  Subset: {subset}")
        print(f"{'='*50}")

        # Path
        test_path  = os.path.join(dataset_root, subset, "bounding_box_test")
        query_path = os.path.join(dataset_root, subset, "query")

        # Load data
        gallery_data = load_images_from_folder(test_path)
        query_data   = load_images_from_folder(query_path)
        print(f"Gallery: {len(gallery_data)} | Query: {len(query_data)}")

        # Ekstrak fitur
        print("Ekstrak fitur gallery...")
        g_feats, g_labels, g_cams = extract_features_from_paths(
            gallery_data, model, transform, device
        )
        
        gallery_cache[model_name][subset] = {
            "feats" : g_feats,
            "labels": g_labels,
            "paths" : [d[0] for d in gallery_data],
        }
        print("Ekstrak fitur query...")
        q_feats, q_labels, q_cams = extract_features_from_paths(
            query_data, model, transform, device
        )

        # Evaluasi
        cmc_base, map_base = evaluate_reid(q_feats, q_labels, g_feats, g_labels)
        cmc_rr,   map_rr   = evaluate_reid_reranking(q_feats, q_labels, g_feats, g_labels)

        all_model_results[model_name][subset] = {
            "baseline" : {
                "r1" : cmc_base[0]*100, "r5" : cmc_base[4]*100,
                "r10": cmc_base[9]*100, "mAP": map_base*100
            },
            "reranking": {
                "r1" : cmc_rr[0]*100,   "r5" : cmc_rr[4]*100,
                "r10": cmc_rr[9]*100,   "mAP": map_rr*100
            },
        }

        print(f"Baseline   → R1: {cmc_base[0]*100:.2f}%  mAP: {map_base*100:.2f}%")
        print(f"Re-ranking → R1: {cmc_rr[0]*100:.2f}%  mAP: {map_rr*100:.2f}%")

    # Bebaskan VRAM setelah setiap model selesai
    models_dict[model_name] = model  # ← tambahkan ini
    
    torch.cuda.empty_cache()

print("\n\nSemua evaluasi selesai!")


######################################################################
  MODEL: Pretrained ImageNet
######################################################################


Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 114MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
Pretrained ImageNet — weight ImageNet digunakan

  Subset: with_bag
Gallery: 1131 | Query: 322
Ekstrak fitur gallery...
  Processed 0/1131
  Processed 320/1131
  Processed 640/1131
  Processed 960/1131
Ekstrak fitur query...
  Processed 0/322
  Processed 320/322
Baseline   → R1: 45.03%  mAP: 12.89%
Re-ranking → R1: 42.24%  mAP: 14.13%

  Subset: without_bag
Gallery: 986 | Query: 179
Ekstrak fitur gallery...
  Processed 0/986
  Processed 320/986
  Processed 640/986
  Processed 960/986
Ekstrak fitur query...
  Processed 0/179
Baseline   → R1: 77.09%  mAP: 47.46%
Re-ranking → R1: 75.42%  mAP: 57.25%

  Subset: both_small
Gallery: 2146 | Query: 475
Ekstrak fitur gallery...
  Processed 0/2146
  Processed 320/2146
  Processed 640/2146
  Processed 960/2146
  Processed 1280/2146
  Processed 1600/2146
  Processed 1920/2146
Ekstrak fitur query...
  Processed 0/475
  Processed 320/475
Bas

In [11]:
for model_name, model_results in all_model_results.items():
    print("\n" + "=" * 75)
    print(f"  MODEL: {model_name}".center(75))
    print("=" * 75)
    print(f"{'Subset':<15} {'Metode':<12} {'Rank-1':>8} {'Rank-5':>8} {'Rank-10':>8} {'mAP':>8}")
    print("-" * 75)

    for subset in subsets:
        r = model_results[subset]

        print(f"{subset:<15} {'Baseline':<12} "
              f"{r['baseline']['r1']:>7.2f}% "
              f"{r['baseline']['r5']:>7.2f}% "
              f"{r['baseline']['r10']:>7.2f}% "
              f"{r['baseline']['mAP']:>7.2f}%")

        print(f"{'':15} {'Re-ranking':<12} "
              f"{r['reranking']['r1']:>7.2f}% "
              f"{r['reranking']['r5']:>7.2f}% "
              f"{r['reranking']['r10']:>7.2f}% "
              f"{r['reranking']['mAP']:>7.2f}%")

        d_r1  = r['reranking']['r1']  - r['baseline']['r1']
        d_r5  = r['reranking']['r5']  - r['baseline']['r5']
        d_r10 = r['reranking']['r10'] - r['baseline']['r10']
        d_map = r['reranking']['mAP'] - r['baseline']['mAP']
        print(f"{'':15} {'Selisih':<12} "
              f"{d_r1:>+7.2f}% {d_r5:>+7.2f}% {d_r10:>+7.2f}% {d_map:>+7.2f}%")
        print("-" * 75)

    print("=" * 75)


                          MODEL: Pretrained ImageNet                       
Subset          Metode         Rank-1   Rank-5  Rank-10      mAP
---------------------------------------------------------------------------
with_bag        Baseline       45.03%   66.15%   75.78%   12.89%
                Re-ranking     42.24%   63.66%   70.50%   14.13%
                Selisih        -2.80%   -2.48%   -5.28%   +1.23%
---------------------------------------------------------------------------
without_bag     Baseline       77.09%   90.50%   95.53%   47.46%
                Re-ranking     75.42%   91.06%   92.18%   57.25%
                Selisih        -1.68%   +0.56%   -3.35%   +9.78%
---------------------------------------------------------------------------
both_small      Baseline       55.58%   72.00%   81.89%   23.35%
                Re-ranking     50.95%   70.53%   76.00%   26.90%
                Selisih        -4.63%   -1.47%   -5.89%   +3.55%
---------------------------------------------

In [ ]:
import pickle

# Simpan gallery cache
with open("/kaggle/working/gallery_cache.pkl", "wb") as f:
    pickle.dump(gallery_cache, f)

# Simpan hasil evaluasi (untuk tabel di Gradio)
with open("/kaggle/working/all_model_results.pkl", "wb") as f:
    pickle.dump(all_model_results, f)

print("Semua file tersimpan!")